# Imports

In [ ]:
import json
import os

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import matplotlib.pyplot as plt
import pandas as pd

from uavloc.data.drone_frames import load_frames
from uavloc.data.multidate import load_date
from uavloc.features.factory import correlator_config
from uavloc.features.segmentation import SegmentationFeatures, load_segmenter
from uavloc.geo import LatLon, haversine_m
from uavloc.matching.correlator import PositionCorrelator
from uavloc.matching.replay import Replay, replay
from uavloc.plotting.correlation import plot_model_comparison
from uavloc.track import experiment_dir

# Params

In [ ]:
EXP = "021-building-loss-from-scratch"
ARCHITECTURE = "unet-resnet34"
LOSSES = ["current", "building", "structure"]
RULE = "gated"                        # "gated" (flight simulator gate) | "ungated"
TRAIN_SEED = 42
STEPS = 2500
NOISE_M = 10.0
N_CASES = 3                           # per kind: best, worst, most disagreement between losses
DEVICE = "mps"
OVERVIEW_MARGIN_M = 50.0
REPLAY_TOL_M = 0.05

# Data

In [ ]:
ARTIFACTS = experiment_dir(EXP, create=False) / "artifacts"
config = json.loads((ARTIFACTS / "correction-config.json").read_text())[RULE]
frames = {f.name: f for f in load_frames(config["frames_dataset"])}
DRAW_KEYS = ["frame", "place", "noise_m", "bearing_deg", "prior_lat", "prior_lon", "prior_error_m"]

In [ ]:
def loss_errors(loss: str) -> pd.DataFrame:
    """Error of every draw at NOISE_M under RULE, as a column named after `loss`."""
    r = pd.read_csv(ARTIFACTS / f"correction-records-{ARCHITECTURE}-{loss}-seed{TRAIN_SEED}.csv")
    r = r[(r["rule"] == RULE) & (r["noise_m"] == NOISE_M)]
    return r[DRAW_KEYS + ["error_m"]].rename(columns={"error_m": loss})

In [ ]:
draws = loss_errors(LOSSES[0])
for loss in LOSSES[1:]:
    draws = draws.merge(loss_errors(loss), on=DRAW_KEYS)
draws["mean_error_m"] = draws[LOSSES].mean(axis=1)
draws["spread_m"] = draws[LOSSES].max(axis=1) - draws[LOSSES].min(axis=1)
len(draws)

In [ ]:
def pick(column: str, largest: bool, kind: str) -> pd.DataFrame:
    """The N_CASES frames with the largest (or smallest) `column`, one draw per frame."""
    ranked = draws.sort_values(column, ascending=not largest).drop_duplicates("frame")
    return ranked.head(N_CASES).assign(kind=kind)

In [ ]:
cases = pd.concat([pick("mean_error_m", False, "best"), pick("mean_error_m", True, "worst"),
                   pick("spread_m", True, "disagree")], ignore_index=True)
cases[["kind", "frame", "bearing_deg", *LOSSES, "spread_m"]].round(2)

# Model

In [ ]:
maps = {place: load_date(place, config["map_date"]) for place in cases["place"].unique()}
correlators = {}
for loss in LOSSES:
    model = load_segmenter(ARTIFACTS / f"model-{ARCHITECTURE}-{loss}-seed{TRAIN_SEED}-steps{STEPS}.pt", DEVICE,
                           ARCHITECTURE)
    for place, satellite in maps.items():
        features = SegmentationFeatures(model, model, satellite, config["seg_channels"], config["drone_seg_gsd_m"],
                                        config["match_gsd_m"], DEVICE)
        correlators[loss, place] = PositionCorrelator(features, correlator_config(config))

# Run

In [ ]:
replays = [{loss: replay(correlators[loss, c.place], frames[c.frame], LatLon(c.prior_lat, c.prior_lon))
            for loss in LOSSES} for c in cases.itertuples()]

In [ ]:
def replay_error_m(r: Replay) -> float:
    return haversine_m(r.correction.position, r.frame.position)

In [ ]:
gap_m = max(abs(replay_error_m(case_replays[loss]) - getattr(case, loss))
            for case, case_replays in zip(cases.itertuples(), replays) for loss in LOSSES)
if gap_m > REPLAY_TOL_M:
    raise RuntimeError(f"replays differ from the correction records by up to {gap_m:.3f} m")
gap_m

# Plots

In [ ]:
out = experiment_dir(EXP) / "figures" / f"cases-{ARCHITECTURE}-{RULE}-at{NOISE_M:g}m"
out.mkdir(parents=True, exist_ok=True)
for case, case_replays in zip(cases.itertuples(), replays):
    rank = case.Index % N_CASES + 1
    errors = ", ".join(f"{loss} {getattr(case, loss):.1f} m" for loss in LOSSES)
    fig = plot_model_comparison(case_replays, maps[case.place],
                                f"{case.kind.upper()} {rank}: {case.frame} | {ARCHITECTURE}, {RULE} | prior "
                                f"{case.prior_error_m:.1f} m off -> {errors}", OVERVIEW_MARGIN_M)
    fig.savefig(out / f"{case.kind}-{rank}-{case.frame}.png", dpi=70)
    plt.show()